In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import sys
import os
import json
import brotli
import io
import subprocess
import numpy as np
from pathlib import Path

from eval_util_and_plots import *

# Initial Setup
Run
```bash
# in prrt-eval/
cargo build --bin main --release && sudo setcap 'cap_net_admin,cap_sys_nice=eip' ./target/release/main
```
in the `prrt-eval` directory. it builds the main.rs, which parses the configs defined below and simulates it. you need to give it permissions to enable thread affinit/priority + tc netem access. This allows python to execute it without needing root.

## Limitations / Troubleshooting
List of notable limitations of the current evaluation setup, read this to either prevent or maybe understand/debug unexpected results.
### Tracing Duration
Currently, `prrt-eval/prrt/prrt/src/constants.rs` defines `TRACE_DURATION` to a default of 240 seconds. This requires ~14GB of memory for pre-allocated buffers. This is necessary to minimize the cost of tracing/noise of it. This also means that experiments should not be longer than 240 seconds. If the trace buffer is filled, it stops writing, meaning you will see the data simlpy stop at some point in a plot.
#### Increasing tracing duration
You may increase or decrease the `TRACE_DURATION` depending on the capabilities of your system and evaluation needs by manually changing the value in `prrt-eval/prrt/prrt/src/constants.rs` and rebuilding the experiment runner, aka re-run `cargo build --bin main --release && sudo setcap 'cap_net_admin,cap_sys_nice=eip' ./target/release/main`.

### Congestion collapse
From my testing/benchmarking of the prrt protocol implementation, the rust code is quite fast. In practice, it is very likely that one hits the limitations of the kernel networking stack first. This might cause occasional artifacts in the evaluations (e.g. loss spikes) or, when the kernel can no longer keep up, cause congestion collapse. This occurs when the experiment sends data too fast for the kernel, which drops packets, which prrt detects as packet loss, and codes more parity packets i response. Since the implementation currently neither has a way to detect nor prevent this from occuring, the loss traces will show a permanently elavated loss rate. To fix this, simply increase the `SLOWDOWN_FACTOR` in the cell below. This maintains the experiment duration, but reducing the packet counts.

### tc-netem Limitations
Unfortunately, changing the loss/delay configurations of the `lo` queues does cause them to drop packets. You will see this as large packet loss spikes around the phase transitions of the experiment configurations. This also means that e.g. simulating drift by repeatedly degrading/improving the loss will not have the desired effect. Furthermore, the `prrt/lltc/` wrapper currently only supports the GE model, which is hardcoded as a Simplified Gilbert Elliot (SGE) (good state= 0% loss, bad state=100% loss) model in the experiment runner. SGE models can simulate IID channels, by setting the correlation to 0.

# Background Information

Create an experiment by defining a list of configurations (`config_sequence`). each entry has the format `(loss_rate, correlation, RTT_ms, target_loss, packet_count)`.
The code will then simulate the experiment and put the traces in `trace_dir`. 

you can modify `trace_dir` to generate you own data instead of using our execution traces. Note that by default, tracing data will not be overwritten, you must manually delete it, or preferably, create a new `trace_dir` path.

In [ ]:
SLOWDOWN_FACTOR = 1

# Format: (loss_rate, correlation, RTT_ms, target_loss, packet_count)
c1 = (0.01, 0.0, 20,  0.001, 400_000/SLOWDOWN_FACTOR)
c2 = (0.05, 0.0, 20,  0.001, 400_000/SLOWDOWN_FACTOR)
c3 = (0.01, 0.7, 20,  0.001, 800_000/SLOWDOWN_FACTOR)
config_sequence = [c1, c2, c3, c1]


# Global Settings
SENDER_TARGET_DELAY_MS = 100
INTER_PACKET_DELAY_US = 20*SLOWDOWN_FACTOR
RUST_BINARY_PATH = "../target/release/main"
trace_dir = "traces/mismatch/" # Change this to something else to generate fresh data!


# Used files, we discard all other files to minimize serializing traces
used_trace_files = [
    trace_dir+"phase_switches.csv",
    trace_dir+"sender/source_packet_send.csv.br",
    trace_dir+"sender/parity_packet_send.csv.br",
    trace_dir+"controller/source_packet_received.csv.br",
    trace_dir+"controller/controller_local_erasure_rate.csv.br",
    trace_dir+"controller/controller_e2e_erasure_rate.csv.br",
    trace_dir+"controller/controller_corrected_erasure_rate.csv.br",
    trace_dir+"controller/controller_schedule_update_ri.csv.br",
    trace_dir+"controller/controller_schedule_update_packet_debt.csv.br",
    trace_dir+"controller/controller_integral_error.csv.br"
]
discard_unused_traces = True

In [ ]:

payload = []
for c in config_sequence:
    payload.append({
        "loss_rate": c[0],
        "correlation": c[1],
        "rtt_ms": int(c[2]),
        "target_erasure_rate": c[3],
        "packet_count": int(c[4])
    })

json_arg = json.dumps(payload)

# --- 2. EXECUTE ---


cmd = [
    RUST_BINARY_PATH,
    "--sender-delay-ms", str(SENDER_TARGET_DELAY_MS),
    "--inter-packet-delay-us", str(INTER_PACKET_DELAY_US),
    "--config-json", json_arg
]

print(f"Starting simulation...")
print(f"Delay: {SENDER_TARGET_DELAY_MS}ms, Pacing: {INTER_PACKET_DELAY_US}us")
print(f"Total Packets: {sum(p['packet_count'] for p in payload)}")

custom_env = os.environ.copy()
custom_env["PRRT_TRACE_DIR"] = trace_dir

if os.path.exists(trace_dir) and len(os.listdir(trace_dir)) > 0:
    print(f"Data already exists in '{trace_dir}'. Skipping simulation.")
else:
    # Ensure the directory exists
    os.makedirs(trace_dir, exist_ok=True)

    # catch missing binary
    if not os.path.exists(RUST_BINARY_PATH):
        print(f"Error: Binary not found at {RUST_BINARY_PATH}")
        print("Run: cargo build --bin main --release && sudo setcap 'cap_net_admin,cap_sys_nice=eip' ./target/release/main")
        sys.exit(1)
    
    print("Starting simulation...")
    print(f"Delay: {SENDER_TARGET_DELAY_MS}ms, Pacing: {INTER_PACKET_DELAY_US}us")
    print(f"Total Packets: {sum(p['packet_count'] for p in payload)}")

    try:
        process = subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL, env=custom_env)
        print("--- Simulation Complete ---")

    except subprocess.CalledProcessError as e:
        print(f"Error executing simulation: {e}")
        sys.exit(1)

if discard_unused_traces:
    keep_files = {Path(f).resolve() for f in used_trace_files}
    directory = Path(trace_dir).resolve()
    if not directory.exists():
        print(f"Error: Directory '{trace_dir}' does not exist.")
    else:
        # Recursively find all files in the directory (handles subfolders like sender/ and controller/)
        for file_path in directory.rglob('*'):
            if file_path.is_file():
                # If the file is not in our 'keep' list, delete it
                if file_path.resolve() not in keep_files:
                    try:
                        file_path.unlink()
                        # print(f"Deleted: {file_path}")
                    except Exception as e:
                        print(f"Error deleting {file_path}: {e}")

In [ ]:
# Paper plot
plot_subplots_erasure_rates_and_RI(trace_dir, config_sequence, save_path="combined_plot.png")

In [ ]:
# No Overlay
plot_erasure_rates(trace_dir, config_sequence)

In [ ]:
# Packet Debt Overlay
plot_erasure_rates(
    trace_dir, 
    config_sequence, 
    overlay_file='controller_schedule_update_packet_debt.csv.br',
    overlay_label='Packet Debt',
    save_path="missmatch_debt.png"
)

In [ ]:
# Integral Term
plot_erasure_rates(
    trace_dir, 
    config_sequence, 
    overlay_file='controller_integral_error.csv.br',
    overlay_label='Integral Term',
    save_path="missmatch_debt_integral.png"
)


In [ ]:
# Convergence Quality
plot_erasure_distribution(trace_dir, 
    config_sequence, save_path="missmatch_erasure_distribution.png")

In [ ]:
# RI plot
plot_redundancy_evolution(trace_dir, config_sequence, save_path="missmatch_redundancy_evolution.png")

In [ ]:
#decompress_brotli("traces/sender/source_packet_send.csv.br")